# `content`, `content_blocks` and `output_version`

This notebook makes no server calls. `GenericFakeChatModel` (from langchain-core) stands in for `ChatLlama`: it returns the `AIMessage` that M1-b produces, and the base class then does exactly what it would do for `ChatLlama`.

The llama-server reply being simulated:
```json
"choices":[{"message":{"role":"assistant",
   "reasoning_content":"User asks capital of France...",
   "content":"Paris."}}]
```

In [1]:
import os

import langchain_core
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

print("langchain-core", langchain_core.__version__)


def server_reply() -> AIMessage:
    """What ChatLlama._generate builds from the reply above (M1-b, option 1)."""
    return AIMessage(
        content="Paris.",
        additional_kwargs={"reasoning_content": "User asks capital of France..."},
    )


def fake_model(**kwargs) -> GenericFakeChatModel:
    return GenericFakeChatModel(messages=iter([server_reply()]), **kwargs)


def show(msg: AIMessage) -> None:
    print(".content           =", repr(msg.content))
    print(".additional_kwargs =", msg.additional_kwargs)
    print(".content_blocks    =", msg.content_blocks)
    print(".text              =", repr(msg.text))
    print(".response_metadata =", msg.response_metadata)

langchain-core 1.6.9


## 1. Default (`output_version=None`, aka v0)

`.content` is whatever the integration stored: a plain string. `.content_blocks` is a **computed property**: it's rebuilt on every read, never stored. With no `model_provider` in `response_metadata`, core's best-effort parser makes the string a `text` block and puts `additional_kwargs["reasoning_content"]` first as a `reasoning` block.

In [2]:
v0 = fake_model().invoke("What is the capital of France?")
show(v0)

.content           = 'Paris.'
.additional_kwargs = {'reasoning_content': 'User asks capital of France...'}
.content_blocks    = [{'type': 'reasoning', 'reasoning': 'User asks capital of France...'}, {'type': 'text', 'text': 'Paris.'}]
.text              = 'Paris.'
.response_metadata = {}


## 2. `output_version='v1'`

The **base class**, not the integration, overwrites `.content` with `.content_blocks` after `_generate` returns, and stamps `response_metadata['output_version']`. Note that `reasoning_content` is still in `additional_kwargs` too: it's duplicated, not moved.

In [3]:
v1 = fake_model(output_version="v1").invoke("What is the capital of France?")
show(v1)

.content           = [{'type': 'reasoning', 'reasoning': 'User asks capital of France...'}, {'type': 'text', 'text': 'Paris.'}]
.additional_kwargs = {'reasoning_content': 'User asks capital of France...'}
.content_blocks    = [{'type': 'reasoning', 'reasoning': 'User asks capital of France...'}, {'type': 'text', 'text': 'Paris.'}]
.text              = 'Paris.'
.response_metadata = {'output_version': 'v1'}


Same switch via the environment: the field's default is `from_env('LC_OUTPUT_VERSION', default=None)`, read at construction.

In [4]:
os.environ["LC_OUTPUT_VERSION"] = "v1"
try:
    model = fake_model()
finally:
    del os.environ["LC_OUTPUT_VERSION"]
print("model.output_version =", repr(model.output_version))
print(".content =", model.invoke("hi").content)

model.output_version = 'v1'
.content = [{'type': 'reasoning', 'reasoning': 'User asks capital of France...'}, {'type': 'text', 'text': 'Paris.'}]


## 3. What changes for calling code

Under v1, `.content` is a list, so string code breaks. `.text` works in both modes. So does `StrOutputParser`, which this cell checks.

In [5]:
from langchain_core.output_parsers import StrOutputParser

for name, msg in [("v0", v0), ("v1", v1)]:
    try:
        upper = msg.content.upper()
    except AttributeError as e:
        upper = f"AttributeError: {e}"
    print(f"{name}: .content.upper() -> {upper}")
    print(f"{name}: .text            -> {msg.text!r}")
    print(f"{name}: StrOutputParser  -> {StrOutputParser().invoke(msg)!r}")

v0: .content.upper() -> PARIS.
v0: .text            -> 'Paris.'
v0: StrOutputParser  -> 'Paris.'
v1: .content.upper() -> AttributeError: 'list' object has no attribute 'upper'
v1: .text            -> 'Paris.'
v1: StrOutputParser  -> 'Paris.'


## 4. Why `ChatLlama` mustn't claim to be `openai`

When `response_metadata['model_provider']` names a provider with a registered translator, `.content_blocks` uses that translator instead of the best-effort parser. Here's the same message, claiming to come from different providers:

In [6]:
for provider in [None, "openai", "llama-cpp"]:
    msg = server_reply()
    if provider:
        msg.response_metadata["model_provider"] = provider
    print(f"model_provider={provider!r:12} -> {msg.content_blocks}")

model_provider=None         -> [{'type': 'reasoning', 'reasoning': 'User asks capital of France...'}, {'type': 'text', 'text': 'Paris.'}]
model_provider='openai'     -> [{'type': 'text', 'text': 'Paris.'}]
model_provider='llama-cpp'  -> [{'type': 'reasoning', 'reasoning': 'User asks capital of France...'}, {'type': 'text', 'text': 'Paris.'}]


## Summary

| | v0 (default) | v1 |
|---|---|---|
| `.content` | `'Paris.'` (string) | list of standard blocks |
| `.content_blocks` | standard blocks (computed) | same |
| `.text` | `'Paris.'` | `'Paris.'` |
| `additional_kwargs['reasoning_content']` | present | still present (duplicated) |
| who does it | integration stores, core parses on read | base class rewrites `.content` after `_generate` |